# NB10 — The Training Loop

**North star callback:** A fine-tuning step is forward → loss → backward → optimizer.
Every optimization in NB02–NB09 reduces one slice of that pipeline. `torch.profiler`
shows where time actually goes. Gradient checkpointing trades extra compute for
dramatically lower VRAM — enabling longer sequences without OOM.

## 1. A minimal training loop

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# Small but realistic decoder: attention + SwiGLU MLP, 4 layers
class MiniDecoderLayer(nn.Module):
    def __init__(self, d=512, heads=8, ffn=2048):
        super().__init__()
        self.norm1  = nn.LayerNorm(d)
        self.norm2  = nn.LayerNorm(d)
        self.q      = nn.Linear(d, d,   bias=False)
        self.k      = nn.Linear(d, d,   bias=False)
        self.v      = nn.Linear(d, d,   bias=False)
        self.o      = nn.Linear(d, d,   bias=False)
        self.gate   = nn.Linear(d, ffn, bias=False)
        self.up     = nn.Linear(d, ffn, bias=False)
        self.down   = nn.Linear(ffn, d, bias=False)
        self.heads  = heads
        self.hd     = d // heads

    def forward(self, x):
        B, N, D = x.shape
        h = self.norm1(x)
        q = self.q(h).view(B, N, self.heads, self.hd).transpose(1, 2)
        k = self.k(h).view(B, N, self.heads, self.hd).transpose(1, 2)
        v = self.v(h).view(B, N, self.heads, self.hd).transpose(1, 2)
        a = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        x = x + self.o(a.transpose(1, 2).reshape(B, N, D))
        h = self.norm2(x)
        x = x + self.down(F.silu(self.gate(h)) * self.up(h))
        return x


class MiniDecoder(nn.Module):
    def __init__(self, d=512, n_layers=4, heads=8, ffn=2048, vocab=32_000):
        super().__init__()
        self.embed   = nn.Embedding(vocab, d)
        self.layers  = nn.ModuleList([MiniDecoderLayer(d, heads, ffn) for _ in range(n_layers)])
        self.norm    = nn.LayerNorm(d)
        self.lm_head = nn.Linear(d, vocab, bias=False)

    def forward(self, tokens):
        x = self.embed(tokens).to(torch.bfloat16)
        for layer in self.layers:
            x = layer(x)
        return self.lm_head(self.norm(x))


torch.manual_seed(0)
D, N_LAYERS, VOCAB = 512, 4, 32_000
model = MiniDecoder(d=D, n_layers=N_LAYERS, vocab=VOCAB).cuda().bfloat16()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)

B, N = 2, 512
tokens = torch.randint(0, VOCAB, (B, N), device='cuda')

n_params = sum(p.numel() for p in model.parameters())
print(f'Model: {n_params/1e6:.1f}M params  ({n_params*2/1024**2:.1f} MB bf16)')
print()

for step in range(5):
    optimizer.zero_grad()
    logits = model(tokens)                    # (B, N, vocab)
    loss   = F.cross_entropy(
        logits[:, :-1].reshape(-1, VOCAB),    # predict next token
        tokens[:, 1:].reshape(-1),
    )
    loss.backward()
    optimizer.step()
    print(f'step {step}: loss={loss.item():.4f}')

Model: 49.6M params  (94.5 MB bf16)

step 0: loss=10.5000
step 1: loss=10.3750
step 2: loss=10.2500
step 3: loss=10.1250
step 4: loss=9.9375


## 2. Profile with torch.profiler

`torch.profiler` records the CUDA kernel timeline for every op in the training step.
The top entries reveal whether the bottleneck is attention, matmul, memory ops, or
optimizer updates — exactly the kind of analysis that guided Unsloth's optimizations.

In [2]:
from torch.profiler import profile, record_function, ProfilerActivity

def one_step():
    optimizer.zero_grad()
    with record_function('forward'):
        logits = model(tokens)
        loss   = F.cross_entropy(logits[:, :-1].reshape(-1, VOCAB), tokens[:, 1:].reshape(-1))
    with record_function('backward'):
        loss.backward()
    with record_function('optimizer'):
        optimizer.step()

# Warmup outside profiler
for _ in range(3):
    one_step()

with profile(
    activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA],
    record_shapes=False,
    with_stack=False,
) as prof:
    for _ in range(3):
        one_step()

events = prof.key_averages()
top = sorted(events, key=lambda e: e.device_time_total, reverse=True)[:12]

print(f'{"Op":50s} {"CUDA ms":>9s} {"Calls":>6s}')
print('-' * 68)
for e in top:
    name = e.key[:50]
    print(f'{name:50s} {e.device_time_total/1000:>9.3f} {e.count:>6d}')

Op                                                   CUDA ms  Calls
--------------------------------------------------------------------
Optimizer.step#AdamW.step                              7.390      3
optimizer                                              7.344      3
Optimizer.step#AdamW.step                              7.344      3
aten::mm                                               5.410    261
forward                                                5.139      3
autograd::engine::evaluate_function: MmBackward0       3.742     87
MmBackward0                                            3.742     87
forward                                                3.009      3
aten::_foreach_addcdiv_                                1.898      3
void at::native::(anonymous namespace)::multi_tens     1.898      9
aten::linear                                           1.668     87
aten::matmul                                           1.668     87


## 3. Gradient checkpointing

Without GC, every layer's intermediate activations (post-attention, post-MLP) stay
in VRAM for the entire backward pass. With GC, only the checkpoint inputs are stored;
activations are recomputed on the backward pass. Cost: ~33% extra compute.
Benefit: activation memory scales with *layers* instead of *layers × sequence length*.

In [3]:
from torch.utils.checkpoint import checkpoint


def forward_no_gc(model, tokens):
    x = model.embed(tokens).to(torch.bfloat16)
    for layer in model.layers:
        x = layer(x)
    return model.lm_head(model.norm(x))


def forward_with_gc(model, tokens):
    x = model.embed(tokens).to(torch.bfloat16)
    for layer in model.layers:
        x = checkpoint(layer, x, use_reentrant=False)
    return model.lm_head(model.norm(x))


def train_step(forward_fn):
    optimizer.zero_grad()
    logits = forward_fn(model, tokens)
    loss   = F.cross_entropy(logits[:, :-1].reshape(-1, VOCAB), tokens[:, 1:].reshape(-1))
    loss.backward()
    optimizer.step()


def peak_vram_delta(fn):
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    baseline = torch.cuda.memory_allocated()
    fn()
    torch.cuda.synchronize()
    return (torch.cuda.max_memory_allocated() - baseline) / 1024**2


# Warmup both paths
for _ in range(2):
    train_step(forward_no_gc)
    train_step(forward_with_gc)

vram_no_gc = peak_vram_delta(lambda: train_step(forward_no_gc))
vram_gc    = peak_vram_delta(lambda: train_step(forward_with_gc))

print(f'Peak VRAM delta — no GC : {vram_no_gc:.0f} MB')
print(f'Peak VRAM delta — std GC: {vram_gc:.0f} MB')
print(f'VRAM saved              : {vram_no_gc - vram_gc:.0f} MB  '
      f'({(vram_no_gc - vram_gc) / vram_no_gc * 100:.1f}%)')

Peak VRAM delta — no GC : 253 MB
Peak VRAM delta — std GC: 160 MB
VRAM saved              : 92 MB  (36.5%)


## 4. Unsloth's gradient checkpointing — custom autograd.Function

Unsloth replaces `torch.utils.checkpoint.checkpoint` with its own
`Unsloth_Gradient_Checkpointer`, a `torch.autograd.Function` subclass.
The key difference: it is `@torch._disable_dynamo`-decorated to prevent
`torch.compile` from tracing through the re-entrant checkpoint boundary,
which avoids graph breaks and keeps the compiled region intact.

In [4]:
import sys
sys.path.insert(0, "../unsloth")
import unsloth  # sets UNSLOTH_IS_PRESENT env var required by unsloth_zoo

import inspect
from unsloth_zoo.gradient_checkpointing import (
    Unsloth_Gradient_Checkpointer,
    unsloth_gradient_checkpoint,
    patch_unsloth_gradient_checkpointing,
)

print('=== Unsloth_Gradient_Checkpointer ===')
print(inspect.getsource(Unsloth_Gradient_Checkpointer))
print()
print('=== unsloth_gradient_checkpoint ===')
print(inspect.getsource(unsloth_gradient_checkpoint))
print()
print('=== patch_unsloth_gradient_checkpointing ===')
print(inspect.getsource(patch_unsloth_gradient_checkpointing))

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
=== Unsloth_Gradient_Checkpointer ===
class Unsloth_Gradient_Checkpointer(torch.autograd.Function):
    """
    All Unsloth Zoo code licensed under LGPLv3
    Same as normal gradient checkpointing but cleaner
    """
    @staticmethod
    @torch_amp_custom_fwd
    def forward(ctx, forward_function, hidden_states, *args):
        with torch.no_grad():
            output = forward_function(hidden_states, *args)
        ctx.save_for_backward(hidden_states)
        ctx.forward_function = forward_function
        ctx.args = args
        return output
    pass

    @staticmethod
    @torch_amp_custom_bwd
    def backward(ctx, dY):
        (hidden_states,) = ctx.saved_tensors
        hidden_states = hidden_states.detach()
        hid

## 5. Latency trade-off benchmark

In [5]:
import sys
sys.path.insert(0, '..')
from utils.benchmark import compare


def forward_unsloth_gc(model, tokens):
    x = model.embed(tokens).to(torch.bfloat16)
    for layer in model.layers:
        # Unsloth_Gradient_Checkpointer expects forward_function to return a 1-tuple,
        # matching HF decoder layers. Wrap our plain nn.Module accordingly.
        (x,) = Unsloth_Gradient_Checkpointer.apply(lambda h, l=layer: (l(h),), x)
    return model.lm_head(model.norm(x))


results = compare(
    fns={
        'no_gc'      : lambda: train_step(forward_no_gc),
        'standard_gc': lambda: train_step(forward_with_gc),
        'unsloth_gc' : lambda: train_step(forward_unsloth_gc),
    },
    notebook='nb10',
    experiment='gradient_checkpointing',
    n_warmup=3, n_repeat=10,
)

vram_no_gc_b = peak_vram_delta(lambda: train_step(forward_no_gc))
vram_gc_b    = peak_vram_delta(lambda: train_step(forward_with_gc))
vram_usgc_b  = peak_vram_delta(lambda: train_step(forward_unsloth_gc))

print(f'{"variant":14s}  {"latency ms":>10s}  {"peak VRAM MB":>13s}')
print('-' * 42)
for (label, r), vram in zip(results.items(), [vram_no_gc_b, vram_gc_b, vram_usgc_b]):
    print(f'{label:14s}  {r.latency_ms:>10.1f}  {vram:>13.0f}')

variant         latency ms   peak VRAM MB
------------------------------------------
no_gc                  6.8            253
standard_gc           10.9            160
unsloth_gc             8.2            160


## 6. Exercises

1. **Scale the sequence length**: repeat the VRAM comparison (no-GC vs GC) for
   N = 256, 512, 1024, 2048. Plot VRAM delta vs N. At what N does GC start to
   pay off on this model?

2. **Add a gradient norm monitor**: insert a `torch.nn.utils.clip_grad_norm_` call
   into the training step and record the global gradient norm each step.
   Profile the added overhead. Where does `clip_grad_norm_` appear in the
   `torch.profiler` output?

3. **Selective checkpointing**: instead of checkpointing every layer, checkpoint only
   every other layer. Measure the VRAM and latency halfway between no-GC and full-GC.
   Is the trade-off linear in the number of checkpointed layers?